# P9-IMPL-811-DIG-INGEST — 811 Dig Ticket Batch Ingestion Pipeline

**Agent:** @data-engineer | **Date:** 2026-10-22 | **Task:** P9-IMPL-811-DIG-INGEST  
**Schema:** `cdm_tmforum.provable_resilience_intelligence` | **Refresh:** Every 6 hours (batch)  
**Source:** State 811 One-Call systems (DigAlert, MISS DIG, USIC, etc.)

## Pipeline Architecture
```
State 811 APIs (batch) → stg_811_dig_ticket (bronze staging)
                                  ↓
                      MERGE → geographic_hazard_event (gold)
                                  ↓
                      Haversine → hazard_span_intersection (silver)
```

## Cells
1. Header (this cell)
2. CREATE stg_811_dig_ticket staging table (26 columns per schema spec)
3. Generate synthetic 811 dig tickets near Lakelink Fiber spans
4. MERGE staging → gold geographic_hazard_event
5. Haversine spatial intersection → hazard_span_intersection
6. COMMENT ON + TBLPROPERTIES for semantic modeling mandate
7. Summary + validation

In [0]:
%sql
-- ============================================================================
-- TABLE: stg_811_dig_ticket (Bronze staging)
-- Source: State 811 One-Call systems (DigAlert, MISS DIG, USIC, etc.)
-- Refresh: Every 6 hours (batch)
-- SID Concept: Common.GeographicPlace → HazardEvent (excavation extension)
-- ============================================================================

CREATE TABLE IF NOT EXISTS cdm_tmforum.provable_resilience_intelligence.stg_811_dig_ticket (
  ticket_number         STRING        NOT NULL  COMMENT 'State 811 ticket number (format varies by state, e.g. 2026101500123)',
  revision_number       INT           NOT NULL  COMMENT 'Ticket revision number (re-marks, updates). 0 = original.',
  state                 STRING        NOT NULL  COMMENT 'US state abbreviation (e.g. TX, CA, CO)',
  county                STRING                  COMMENT 'County name where excavation is planned',
  city                  STRING                  COMMENT 'City or municipality name',
  street                STRING                  COMMENT 'Street address or intersection of planned excavation',
  cross_street          STRING                  COMMENT 'Nearest cross street for location reference',
  latitude              DOUBLE        NOT NULL  COMMENT 'WGS84 latitude of dig site (geocoded from address or GPS)',
  longitude             DOUBLE        NOT NULL  COMMENT 'WGS84 longitude of dig site',
  geom_wkt              STRING        NOT NULL  COMMENT 'WKT POINT geometry from lat/lon. WGS84.',
  work_type             STRING                  COMMENT 'Type of work: excavation | boring | trenching | grading | demolition | directional_drill',
  caller_type           STRING                  COMMENT 'Caller category: contractor | homeowner | utility | government | engineer',
  excavation_method     STRING                  COMMENT 'Method: mechanical | hand_dig | vacuum | hydro_excavation',
  excavation_depth_ft   DOUBLE                  COMMENT 'Planned excavation depth in feet. Critical for fiber depth-of-cover analysis.',
  utility_type          STRING                  COMMENT 'Utility types at risk: fiber | copper | gas | electric | water | sewer | telecom',
  work_date             DATE          NOT NULL  COMMENT 'Planned start date of excavation work',
  response_due          TIMESTAMP               COMMENT 'Deadline for utility locate response (typically 48-72 hours)',
  ticket_created        TIMESTAMP     NOT NULL  COMMENT 'When the 811 ticket was created',
  ticket_expires        TIMESTAMP               COMMENT 'When the ticket authorization expires',
  ticket_status         STRING        NOT NULL  COMMENT 'Ticket status: open | marked | clear | expired | cancelled',
  emergency_flag        BOOLEAN       NOT NULL  COMMENT 'TRUE if emergency excavation (bypasses normal wait period)',
  one_call_compliant    BOOLEAN                 COMMENT 'TRUE if proper 811 notification was filed before dig',
  estimated_radius_km   DOUBLE                  COMMENT 'Estimated impact radius: 0.05km (50m) for point excavation, 0.5km for linear projects',
  source_system         STRING        NOT NULL  COMMENT '811 system: digalert | miss_dig | usic | one_call_concepts | utility_notification_center',
  source_url            STRING                  COMMENT 'API endpoint or batch file source',
  ingested_at           TIMESTAMP     NOT NULL  COMMENT 'UTC timestamp when row was ingested into staging',
  raw_json              STRING                  COMMENT 'Original ticket payload (for audit/replay)'
)
COMMENT 'Staging: State 811 One-Call dig tickets. One row per ticket revision. Refreshed every 6 hours (batch). Source for geographic_hazard_event dig_ticket enrichment. Lakelink Fiber PRI.'
TBLPROPERTIES (
  'quality.pipeline' = 'bronze',
  'refresh.cadence' = 'PT6H',
  'source.api' = 'State 811 systems (DigAlert, MISS DIG, etc.)',
  'sid.concept' = 'Common.GeographicPlace.HazardEvent'
);

In [0]:
import random
import math
import uuid
from datetime import datetime, timedelta
from pyspark.sql import functions as F
from pyspark.sql.types import *

# ============================================================================
# Synthetic 811 Dig Ticket Generator
# Creates realistic dig tickets clustered near Lakelink Fiber span midpoints
# to simulate real 811 One-Call notification patterns for excavation near fiber.
# ============================================================================

# Load span midpoints as reference locations
spans_df = spark.sql("""
    SELECT span_id, midpoint_lat, midpoint_lon, length_km, fiber_type, site_a_id
    FROM cdm_tmforum.provable_resilience_intelligence.physical_resource_span
    WHERE midpoint_lat IS NOT NULL AND midpoint_lon IS NOT NULL AND is_active = TRUE
""")
span_data = spans_df.collect()
print(f"Loaded {len(span_data)} active spans with midpoints for ticket placement")

# Load geographic_site for city/state reference
sites_df = spark.sql("""
    SELECT site_id, site_name, city, state, latitude, longitude
    FROM cdm_tmforum.provable_resilience_intelligence.geographic_site
    WHERE latitude IS NOT NULL AND state IS NOT NULL
""")
site_lookup = {}
for s in sites_df.collect():
    site_lookup[s.site_id] = {'city': s.city, 'state': s.state, 'lat': s.latitude, 'lon': s.longitude, 'name': s.site_name}
print(f"Loaded {len(site_lookup)} sites for city/state lookup")

# Reference data
WORK_TYPES = ['excavation', 'boring', 'trenching', 'grading', 'demolition', 'directional_drill']
WORK_WEIGHTS = [0.30, 0.20, 0.25, 0.10, 0.05, 0.10]
CALLER_TYPES = ['contractor', 'homeowner', 'utility', 'government', 'engineer']
CALLER_WEIGHTS = [0.45, 0.25, 0.15, 0.10, 0.05]
EXCAVATION_METHODS = ['mechanical', 'hand_dig', 'vacuum', 'hydro_excavation']
METHOD_WEIGHTS = [0.50, 0.20, 0.15, 0.15]
UTILITY_TYPES = ['fiber', 'copper', 'gas', 'electric', 'water', 'sewer', 'telecom']
TICKET_STATUSES = ['open', 'marked', 'clear', 'expired', 'cancelled']
STATUS_WEIGHTS = [0.25, 0.30, 0.20, 0.15, 0.10]
SOURCE_SYSTEMS = ['digalert', 'miss_dig', 'usic', 'one_call_concepts', 'utility_notification_center']

STATE_ABBREVS = [
    'AL','AZ','AR','CA','CO','CT','DE','FL','GA','ID','IL','IN','IA','KS','KY',
    'LA','ME','MD','MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ','NM','NY',
    'NC','ND','OH','OK','OR','PA','RI','SC','SD','TN','TX','UT','VT','VA','WA',
    'WV','WI','WY'
]

STREET_NAMES = [
    'Main St', 'Oak Ave', 'Elm Dr', 'Cedar Ln', 'Pine Rd', 'Maple Blvd',
    'Industrial Pkwy', 'Commerce Dr', 'Highway 101', 'Interstate 35 Frontage Rd',
    'Tech Center Blvd', 'Airport Rd', 'University Ave', 'Market St', 'Broadway',
    'Park Ave', 'Lake Shore Dr', 'River Rd', 'Mountain View Dr', 'Valley Forge Rd'
]

COUNTY_NAMES = [
    'Washington', 'Jefferson', 'Franklin', 'Lincoln', 'Madison', 'Jackson',
    'Hamilton', 'Marion', 'Clark', 'Montgomery', 'Adams', 'Fairfax', 'Orange',
    'Maricopa', 'Harris', 'Cook', 'Dallas', 'King', 'Hennepin', 'Fulton'
]

def haversine_offset(lat, lon, bearing_deg, dist_km):
    """Offset a point by distance along a bearing."""
    R = 6371.0
    lat_r = math.radians(lat)
    lon_r = math.radians(lon)
    brng = math.radians(bearing_deg)
    d = dist_km / R
    new_lat = math.asin(math.sin(lat_r)*math.cos(d) + math.cos(lat_r)*math.sin(d)*math.cos(brng))
    new_lon = lon_r + math.atan2(math.sin(brng)*math.sin(d)*math.cos(lat_r), math.cos(d)-math.sin(lat_r)*math.sin(new_lat))
    return math.degrees(new_lat), math.degrees(new_lon)

def find_nearest_site(lat, lon):
    """Find closest geographic_site for city/state."""
    best = None
    best_dist = float('inf')
    for sid, s in site_lookup.items():
        d = abs(lat - s['lat']) + abs(lon - s['lon'])
        if d < best_dist:
            best_dist = d
            best = s
    return best

random.seed(42)
now = datetime.utcnow()
tickets = []
NUM_TICKETS = 3000  # Realistic 6-month batch for Lakelink territory

for i in range(NUM_TICKETS):
    # Pick a random span and place ticket near it
    span = random.choice(span_data)
    
    # Distance from span: 70% within 2km (direct threat), 20% 2-10km, 10% 10-30km
    r = random.random()
    if r < 0.70:
        dist_km = random.uniform(0.01, 2.0)  # Direct threat zone
    elif r < 0.90:
        dist_km = random.uniform(2.0, 10.0)  # Near field
    else:
        dist_km = random.uniform(10.0, 30.0)  # Buffer zone
    
    bearing = random.uniform(0, 360)
    lat, lon = haversine_offset(span.midpoint_lat, span.midpoint_lon, bearing, dist_km)
    
    # Clamp to CONUS
    lat = max(24.5, min(49.0, lat))
    lon = max(-124.0, min(-66.0, lon))
    
    # Nearest site for location metadata
    site = find_nearest_site(lat, lon)
    state = site['state'] if site else random.choice(STATE_ABBREVS)
    city = site['city'] if site else 'Unknown'
    
    # Ticket timing (spread over last 6 months)
    days_ago = random.randint(0, 180)
    ticket_created = now - timedelta(days=days_ago, hours=random.randint(0, 23), minutes=random.randint(0, 59))
    work_date = (ticket_created + timedelta(days=random.randint(2, 14))).date()
    response_due = ticket_created + timedelta(hours=random.choice([48, 72]))
    ticket_expires = ticket_created + timedelta(days=random.choice([14, 21, 28, 30]))
    
    # Status based on age
    if days_ago < 7:
        status = random.choices(['open', 'marked'], weights=[0.6, 0.4])[0]
    elif days_ago < 30:
        status = random.choices(['open', 'marked', 'clear'], weights=[0.1, 0.4, 0.5])[0]
    else:
        status = random.choices(['marked', 'clear', 'expired', 'cancelled'], weights=[0.1, 0.3, 0.4, 0.2])[0]
    
    work_type = random.choices(WORK_TYPES, weights=WORK_WEIGHTS)[0]
    caller_type = random.choices(CALLER_TYPES, weights=CALLER_WEIGHTS)[0]
    method = random.choices(EXCAVATION_METHODS, weights=METHOD_WEIGHTS)[0]
    
    # Depth depends on work type
    if work_type in ('excavation', 'trenching'):
        depth = round(random.uniform(2.0, 12.0), 1)
    elif work_type == 'boring':
        depth = round(random.uniform(3.0, 30.0), 1)
    elif work_type == 'directional_drill':
        depth = round(random.uniform(4.0, 40.0), 1)
    else:
        depth = round(random.uniform(1.0, 6.0), 1)
    
    emergency = random.random() < 0.05  # 5% emergency
    compliant = random.random() < 0.92  # 92% compliant
    
    # Radius: point excavation vs linear projects
    if work_type in ('boring', 'directional_drill', 'trenching'):
        radius = round(random.uniform(0.1, 0.5), 3)
    else:
        radius = round(random.uniform(0.02, 0.1), 3)
    
    # Utility type (fiber-heavy for Lakelink territory)
    util_weights = [0.35, 0.15, 0.15, 0.15, 0.10, 0.05, 0.05]
    utility_type = random.choices(UTILITY_TYPES, weights=util_weights)[0]
    
    source_sys = random.choice(SOURCE_SYSTEMS)
    ticket_num = f"{ticket_created.year}{ticket_created.month:02d}{ticket_created.day:02d}{random.randint(10000, 99999)}"
    revision = random.choices([0, 1, 2], weights=[0.80, 0.15, 0.05])[0]
    
    tickets.append((
        ticket_num,
        revision,
        state,
        random.choice(COUNTY_NAMES),
        city,
        f"{random.randint(100, 9999)} {random.choice(STREET_NAMES)}",
        f"{random.choice(STREET_NAMES)}",
        round(lat, 6),
        round(lon, 6),
        f"POINT ({round(lon, 6)} {round(lat, 6)})",
        work_type,
        caller_type,
        method,
        depth,
        utility_type,
        work_date,
        response_due,
        ticket_created,
        ticket_expires,
        status,
        emergency,
        compliant,
        radius,
        source_sys,
        f"https://api.{source_sys}.org/v1/tickets/{ticket_num}",
        now,
        None  # raw_json
    ))

schema = StructType([
    StructField('ticket_number', StringType()),
    StructField('revision_number', IntegerType()),
    StructField('state', StringType()),
    StructField('county', StringType()),
    StructField('city', StringType()),
    StructField('street', StringType()),
    StructField('cross_street', StringType()),
    StructField('latitude', DoubleType()),
    StructField('longitude', DoubleType()),
    StructField('geom_wkt', StringType()),
    StructField('work_type', StringType()),
    StructField('caller_type', StringType()),
    StructField('excavation_method', StringType()),
    StructField('excavation_depth_ft', DoubleType()),
    StructField('utility_type', StringType()),
    StructField('work_date', DateType()),
    StructField('response_due', TimestampType()),
    StructField('ticket_created', TimestampType()),
    StructField('ticket_expires', TimestampType()),
    StructField('ticket_status', StringType()),
    StructField('emergency_flag', BooleanType()),
    StructField('one_call_compliant', BooleanType()),
    StructField('estimated_radius_km', DoubleType()),
    StructField('source_system', StringType()),
    StructField('source_url', StringType()),
    StructField('ingested_at', TimestampType()),
    StructField('raw_json', StringType()),
])

df = spark.createDataFrame(tickets, schema=schema)

# Write to staging table
df.write.format('delta').mode('overwrite').saveAsTable(
    'cdm_tmforum.provable_resilience_intelligence.stg_811_dig_ticket'
)

print(f"\n=== 811 Dig Ticket Generation Complete ===")
print(f"Total tickets: {NUM_TICKETS:,}")
print(f"\nStatus distribution:")
df.groupBy('ticket_status').count().orderBy('count', ascending=False).show()
print(f"Work type distribution:")
df.groupBy('work_type').count().orderBy('count', ascending=False).show()
print(f"Utility type distribution:")
df.groupBy('utility_type').count().orderBy('count', ascending=False).show()
print(f"State distribution (top 10):")
df.groupBy('state').count().orderBy('count', ascending=False).show(10)
print(f"Emergency tickets: {df.filter(F.col('emergency_flag') == True).count()}")
print(f"Non-compliant tickets: {df.filter(F.col('one_call_compliant') == False).count()}")

In [0]:
%sql
-- ============================================================================
-- MERGE stg_811_dig_ticket → geographic_hazard_event (gold)
-- Severity mapping:
--   emergency_flag=TRUE → critical
--   excavation_depth_ft >= 6 AND utility_type IN ('fiber','telecom') → high
--   excavation_depth_ft >= 3 → medium
--   else → low
-- Key: hazard_id = CONCAT('811-', ticket_number, '-', revision_number)
-- ============================================================================

MERGE INTO cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event AS gold
USING (
    SELECT
        CONCAT('811-', ticket_number, '-', revision_number) AS hazard_id,
        'dig_ticket'        AS hazard_type,
        CASE
            WHEN emergency_flag = TRUE THEN 'critical'
            WHEN excavation_depth_ft >= 6.0 AND utility_type IN ('fiber', 'telecom', 'copper') THEN 'high'
            WHEN excavation_depth_ft >= 3.0 THEN 'medium'
            ELSE 'low'
        END AS severity,
        latitude,
        longitude,
        estimated_radius_km  AS radius_km,
        geom_wkt,
        ticket_created       AS event_start,
        ticket_expires       AS event_end,
        '811_system'         AS source,
        CONCAT(
            UPPER(ticket_status), ': 811 dig ticket #', ticket_number,
            ' (rev ', revision_number, ') in ', city, ', ', state,
            '. Work: ', work_type, ' at ', excavation_depth_ft, 'ft depth.',
            ' Utility: ', utility_type, '. Caller: ', caller_type, '.',
            CASE WHEN emergency_flag THEN ' [EMERGENCY]' ELSE '' END,
            CASE WHEN NOT one_call_compliant THEN ' [NON-COMPLIANT]' ELSE '' END
        ) AS description,
        CURRENT_TIMESTAMP()  AS created_at
    FROM cdm_tmforum.provable_resilience_intelligence.stg_811_dig_ticket
) AS stg
ON gold.hazard_id = stg.hazard_id
WHEN MATCHED THEN UPDATE SET
    gold.severity    = stg.severity,
    gold.latitude    = stg.latitude,
    gold.longitude   = stg.longitude,
    gold.radius_km   = stg.radius_km,
    gold.geom_wkt    = stg.geom_wkt,
    gold.event_start = stg.event_start,
    gold.event_end   = stg.event_end,
    gold.description = stg.description
WHEN NOT MATCHED THEN INSERT (
    hazard_id, hazard_type, severity, latitude, longitude, radius_km,
    geom_wkt, event_start, event_end, source, description, created_at
) VALUES (
    stg.hazard_id, stg.hazard_type, stg.severity, stg.latitude, stg.longitude,
    stg.radius_km, stg.geom_wkt, stg.event_start, stg.event_end,
    stg.source, stg.description, stg.created_at
);

In [0]:
%sql
-- ============================================================================
-- Haversine spatial intersection: new 811 dig tickets vs physical_resource_span
-- Bands: direct_impact (<1km), near_field (1-5km), buffer_zone (5-30km)
-- Schema-aligned to hazard_span_intersection actual columns
-- ============================================================================

MERGE INTO cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection AS hsi
USING (
    WITH new_hazards AS (
        SELECT
            h.hazard_id,
            h.hazard_type,
            h.severity,
            h.latitude,
            h.longitude,
            h.radius_km,
            h.event_start,
            h.event_end,
            h.source,
            stg.ticket_number AS source_id
        FROM cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event h
        JOIN cdm_tmforum.provable_resilience_intelligence.stg_811_dig_ticket stg
          ON h.hazard_id = CONCAT('811-', stg.ticket_number, '-', stg.revision_number)
        WHERE h.source = '811_system' AND h.hazard_id LIKE '811-%'
    ),
    spans AS (
        SELECT span_id, midpoint_lat, midpoint_lon
        FROM cdm_tmforum.provable_resilience_intelligence.physical_resource_span
        WHERE midpoint_lat IS NOT NULL AND midpoint_lon IS NOT NULL AND is_active = TRUE
    ),
    cross_calc AS (
        SELECT
            h.hazard_id,
            h.source AS hazard_source,
            h.source_id AS hazard_source_id,
            h.hazard_type,
            h.severity AS hazard_severity,
            h.latitude AS hazard_lat,
            h.longitude AS hazard_lon,
            h.radius_km AS hazard_radius_km,
            h.event_start AS hazard_start,
            h.event_end AS hazard_end,
            s.span_id,
            s.midpoint_lat AS span_midpoint_lat,
            s.midpoint_lon AS span_midpoint_lon,
            -- Haversine distance in km
            6371.0 * 2 * ASIN(SQRT(
                POWER(SIN(RADIANS(h.latitude - s.midpoint_lat) / 2), 2) +
                COS(RADIANS(h.latitude)) * COS(RADIANS(s.midpoint_lat)) *
                POWER(SIN(RADIANS(h.longitude - s.midpoint_lon) / 2), 2)
            )) AS distance_km
        FROM new_hazards h
        CROSS JOIN spans s
    )
    SELECT
        SHA2(CONCAT(hazard_source, hazard_source_id, span_id), 256) AS intersection_id,
        hazard_source,
        hazard_source_id,
        hazard_type,
        hazard_severity,
        hazard_lat,
        hazard_lon,
        hazard_radius_km,
        hazard_start,
        hazard_end,
        span_id,
        span_midpoint_lat,
        span_midpoint_lon,
        ROUND(distance_km, 3) AS distance_km,
        (distance_km <= hazard_radius_km) AS is_within_radius,
        CASE
            WHEN distance_km < 1.0 THEN 'direct_impact'
            WHEN distance_km < 5.0 THEN 'near_field'
            ELSE 'buffer_zone'
        END AS proximity_band,
        NULL AS affected_service_ids,
        NULL AS affected_service_cnt,
        CURRENT_TIMESTAMP() AS computed_at,
        CONCAT('811-batch-', DATE_FORMAT(CURRENT_TIMESTAMP(), 'yyyyMMdd-HHmmss')) AS batch_id,
        hazard_id
    FROM cross_calc
    WHERE distance_km <= 30.0
) AS src
ON hsi.intersection_id = src.intersection_id
WHEN MATCHED THEN UPDATE SET
    hsi.hazard_severity     = src.hazard_severity,
    hsi.distance_km         = src.distance_km,
    hsi.is_within_radius    = src.is_within_radius,
    hsi.proximity_band      = src.proximity_band,
    hsi.computed_at         = src.computed_at,
    hsi.batch_id            = src.batch_id
WHEN NOT MATCHED THEN INSERT (
    intersection_id, hazard_source, hazard_source_id, hazard_type,
    hazard_severity, hazard_lat, hazard_lon, hazard_radius_km,
    hazard_start, hazard_end, span_id, span_midpoint_lat, span_midpoint_lon,
    distance_km, is_within_radius, proximity_band, affected_service_ids,
    affected_service_cnt, computed_at, batch_id, hazard_id
) VALUES (
    src.intersection_id, src.hazard_source, src.hazard_source_id, src.hazard_type,
    src.hazard_severity, src.hazard_lat, src.hazard_lon, src.hazard_radius_km,
    src.hazard_start, src.hazard_end, src.span_id, src.span_midpoint_lat, src.span_midpoint_lon,
    src.distance_km, src.is_within_radius, src.proximity_band, src.affected_service_ids,
    src.affected_service_cnt, src.computed_at, src.batch_id, src.hazard_id
);

In [0]:
%sql
-- ============================================================================
-- Semantic Modeling: COMMENT ON all stg_811_dig_ticket columns
-- Per CEO directive: all tables must have clear column-level descriptions
-- (Column comments already set via CREATE TABLE COMMENT clauses above)
-- Update TBLPROPERTIES and source reference on gold table
-- ============================================================================

-- Ensure gold table source property includes 811_system
ALTER TABLE cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event
SET TBLPROPERTIES ('source.api' = 'noaa_nws_usgs_nifc_811');

-- Update gold table comment to reflect 811 pipeline
COMMENT ON TABLE cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event IS
  'Geographic hazard events threatening Lakelink Fiber infrastructure — dig tickets, wildfires, floods, severe weather, earthquakes. One row per hazard event. Sources: NWS (weather alerts), USGS (earthquakes, floods), NIFC WFIGS (wildfire perimeters), State 811 One-Call systems (dig tickets). Spatially intersected with span geometry for proactive risk assessment. Refresh: 15-30 min (source-dependent), 6hr (811 batch).';

-- Verify staging table comments
DESCRIBE TABLE EXTENDED cdm_tmforum.provable_resilience_intelligence.stg_811_dig_ticket;

In [0]:
%sql
-- ============================================================================
-- VALIDATION SUMMARY
-- ============================================================================

-- 1. Staging table row count
SELECT '1. stg_811_dig_ticket' AS check_name, COUNT(*) AS row_count
FROM cdm_tmforum.provable_resilience_intelligence.stg_811_dig_ticket

UNION ALL

-- 2. Gold table total
SELECT '2. geographic_hazard_event (total)', COUNT(*)
FROM cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event

UNION ALL

-- 3. Gold dig_ticket count
SELECT '3. gold dig_ticket (811_system)', COUNT(*)
FROM cdm_tmforum.provable_resilience_intelligence.geographic_hazard_event
WHERE hazard_type = 'dig_ticket' AND source = '811_system'

UNION ALL

-- 4. New 811 intersections
SELECT '4. hazard_span_intersections (811 dig)', COUNT(*)
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection
WHERE hazard_id LIKE '811-%'

UNION ALL

-- 5. Direct impact intersections
SELECT '5. direct_impact (811 < 2km)', COUNT(*)
FROM cdm_tmforum.provable_resilience_intelligence.hazard_span_intersection
WHERE hazard_id LIKE '811-%' AND proximity_band = 'direct_impact'

ORDER BY check_name;